# 02. Data Quality Audit — Dunnhumby

Notebook kiểm tra chất lượng của toàn bộ bảng nguồn trước khi tạo feature. Kết quả phân biệt **invalid** (vi phạm khóa/ràng buộc rõ ràng) và **review** (bất thường nghiệp vụ cần xác nhận, ví dụ `QUANTITY = 0`). Notebook chỉ audit, không tự động xoá hay sửa dữ liệu.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import polars as pl

from propensity.io import scan_csv
from propensity.schema.dunnhumby import (
    CAUSAL_SCHEMA_OVERRIDES,
    TRANSACTION_SCHEMA_OVERRIDES,
)

pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(30)

## 1. Load toàn bộ bảng bằng lazy scan

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw" / "Dunnhumby"
tables = {
    "transaction": scan_csv(
        raw_dir / "transaction_data.csv",
        schema_overrides=TRANSACTION_SCHEMA_OVERRIDES,
    ),
    "product": scan_csv(raw_dir / "product.csv"),
    "demographic": scan_csv(raw_dir / "hh_demographic.csv"),
    "campaign_table": scan_csv(raw_dir / "campaign_table.csv"),
    "campaign_desc": scan_csv(raw_dir / "campaign_desc.csv"),
    "coupon": scan_csv(raw_dir / "coupon.csv"),
    "coupon_redemption": scan_csv(raw_dir / "coupon_redempt.csv"),
    "causal": scan_csv(
        raw_dir / "causal_data.csv",
        schema_overrides=CAUSAL_SCHEMA_OVERRIDES,
    ),
}
list(tables)

## 2. Tổng quan row, column, null và exact duplicate

Cell này quét toàn bộ từng file. `causal_data.csv` lớn nên sẽ mất nhiều thời gian hơn các bảng còn lại.

In [ ]:
def table_quality_overview(name: str, frame: pl.LazyFrame) -> pl.DataFrame:
    columns = frame.collect_schema().names()
    return (
        frame.select(
            pl.lit(name).alias("table"),
            pl.len().alias("n_rows"),
            pl.lit(len(columns)).alias("n_columns"),
            pl.sum_horizontal(
                [pl.col(column).null_count() for column in columns]
            ).alias("n_null_cells"),
            pl.struct(columns).n_unique().alias("n_unique_rows"),
        )
        .with_columns(
            (pl.col("n_rows") - pl.col("n_unique_rows")).alias(
                "n_exact_duplicate_rows"
            )
        )
        .collect(engine="streaming")
    )

quality_overview = pl.concat(
    [table_quality_overview(name, frame) for name, frame in tables.items()]
).sort("table")
quality_overview

## 3. Null theo từng cột

In [ ]:
def null_profile(name: str, frame: pl.LazyFrame) -> pl.DataFrame:
    columns = frame.collect_schema().names()
    row_count = frame.select(pl.len()).collect(engine="streaming").item()
    counts = frame.select(
        [pl.col(column).null_count().alias(column) for column in columns]
    ).collect(engine="streaming")
    return pl.DataFrame(
        {
            "table": [name] * len(columns),
            "column": columns,
            "null_count": [counts[column][0] for column in columns],
            "null_share": [
                counts[column][0] / row_count if row_count else None
                for column in columns
            ],
        }
    )

null_summary = pl.concat(
    [null_profile(name, frame) for name, frame in tables.items()]
).sort("null_share", descending=True)
null_summary.filter(pl.col("null_count") > 0)

## 4. Duplicate theo grain kỳ vọng

`duplicate_key_groups` là số nhóm khóa xuất hiện hơn một lần; `extra_rows` là số dòng dư ngoài một dòng đại diện cho mỗi khóa.

In [ ]:
expected_grains = {
    "transaction": ["BASKET_ID", "PRODUCT_ID"],
    "product": ["PRODUCT_ID"],
    "demographic": ["household_key"],
    "campaign_table": ["CAMPAIGN", "household_key"],
    "campaign_desc": ["CAMPAIGN"],
    "coupon": ["COUPON_UPC", "PRODUCT_ID", "CAMPAIGN"],
    "coupon_redemption": [
        "household_key", "DAY", "COUPON_UPC", "CAMPAIGN"
    ],
    "causal": ["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
}

def duplicate_profile(
    name: str, frame: pl.LazyFrame, keys: list[str]
) -> pl.DataFrame:
    duplicate_groups = (
        frame.group_by(keys)
        .agg(pl.len().alias("rows_per_key"))
        .filter(pl.col("rows_per_key") > 1)
    )
    return duplicate_groups.select(
        pl.lit(name).alias("table"),
        pl.lit(" + ".join(keys)).alias("expected_grain"),
        pl.len().alias("duplicate_key_groups"),
        (pl.col("rows_per_key").sum() - pl.len()).alias("extra_rows"),
        pl.col("rows_per_key").max().alias("max_rows_per_key"),
    ).collect(engine="streaming")

duplicate_summary = pl.concat(
    [
        duplicate_profile(name, tables[name], keys)
        for name, keys in expected_grains.items()
    ]
).sort("extra_rows", descending=True)
duplicate_summary

In [ ]:
# Đổi tên bảng tại đây để xem tối đa 20 nhóm duplicate cụ thể.
duplicate_table = "transaction"
duplicate_keys = expected_grains[duplicate_table]
(
    tables[duplicate_table]
    .group_by(duplicate_keys)
    .agg(pl.len().alias("rows_per_key"))
    .filter(pl.col("rows_per_key") > 1)
    .sort("rows_per_key", descending=True)
    .limit(20)
    .collect(engine="streaming")
)

## 5. Kiểm tra giá trị giao dịch bất hợp lệ hoặc đáng nghi

`QUANTITY = 0` được xếp vào `review`, không mặc định là lỗi. `SALES_VALUE = 0` với quantity dương vẫn là purchase hợp lệ theo định nghĩa target hiện tại.

In [ ]:
transaction = tables["transaction"]
discount_total = pl.sum_horizontal(
    [
        -pl.col("RETAIL_DISC").fill_null(0.0),
        -pl.col("COUPON_DISC").fill_null(0.0),
        -pl.col("COUPON_MATCH_DISC").fill_null(0.0),
    ]
)
transaction_rules = {
    "null_business_key": (
        pl.col("household_key").is_null()
        | pl.col("BASKET_ID").is_null()
        | pl.col("PRODUCT_ID").is_null()
        | pl.col("DAY").is_null()
    ),
    "quantity_zero": pl.col("QUANTITY") == 0,
    "quantity_negative": pl.col("QUANTITY") < 0,
    "sales_negative": pl.col("SALES_VALUE") < 0,
    "positive_quantity_zero_sales": (
        (pl.col("QUANTITY") > 0) & (pl.col("SALES_VALUE") == 0)
    ),
    "zero_quantity_nonzero_sales": (
        (pl.col("QUANTITY") == 0) & (pl.col("SALES_VALUE") != 0)
    ),
    "positive_discount_field": (
        (pl.col("RETAIL_DISC").fill_null(0) > 0)
        | (pl.col("COUPON_DISC").fill_null(0) > 0)
        | (pl.col("COUPON_MATCH_DISC").fill_null(0) > 0)
    ),
    "discount_exceeds_sales": discount_total > pl.col("SALES_VALUE"),
    "invalid_day_or_week": (pl.col("DAY") < 1) | (pl.col("WEEK_NO") < 1),
    "invalid_hhmm_time": (
        (pl.col("TRANS_TIME") // 100 > 23)
        | (pl.col("TRANS_TIME") % 100 > 59)
        | (pl.col("TRANS_TIME") < 0)
    ),
}
rule_severity = {
    "null_business_key": "invalid",
    "quantity_zero": "review",
    "quantity_negative": "review_return",
    "sales_negative": "review_return",
    "positive_quantity_zero_sales": "valid_special_case",
    "zero_quantity_nonzero_sales": "review",
    "positive_discount_field": "review_sign_convention",
    "discount_exceeds_sales": "review",
    "invalid_day_or_week": "invalid",
    "invalid_hhmm_time": "invalid",
}

transaction_rule_summary = transaction.select(
    pl.len().alias("n_rows"),
    *[condition.sum().alias(rule) for rule, condition in transaction_rules.items()],
).collect(engine="streaming")
transaction_rule_summary = pl.DataFrame(
    {
        "rule": list(transaction_rules),
        "severity": [rule_severity[rule] for rule in transaction_rules],
        "n_rows": [
            transaction_rule_summary[rule][0] for rule in transaction_rules
        ],
        "share": [
            transaction_rule_summary[rule][0]
            / transaction_rule_summary["n_rows"][0]
            for rule in transaction_rules
        ],
    }
).sort("n_rows", descending=True)
transaction_rule_summary

In [ ]:
# Đổi rule để kiểm tra raw rows; mặc định xem QUANTITY = 0.
selected_rule = "quantity_zero"
(
    transaction.filter(transaction_rules[selected_rule])
    .select(
        "household_key", "BASKET_ID", "DAY", "WEEK_NO",
        "PRODUCT_ID", "QUANTITY", "SALES_VALUE",
        "RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC",
    )
    .limit(50)
    .collect(engine="streaming")
)

## 6. Domain checks cho các bảng phụ

In [ ]:
domain_rules = {
    "product_nonpositive_id": ("product", pl.col("PRODUCT_ID") <= 0),
    "product_blank_category": (
        "product",
        pl.any_horizontal(
            [
                pl.col(column).is_null() | (pl.col(column).str.strip_chars() == "")
                for column in [
                    "DEPARTMENT", "BRAND", "COMMODITY_DESC",
                    "SUB_COMMODITY_DESC",
                ]
            ]
        ),
    ),
    "campaign_start_after_end": (
        "campaign_desc", pl.col("START_DAY") > pl.col("END_DAY")
    ),
    "campaign_nonpositive_day": (
        "campaign_desc", (pl.col("START_DAY") < 1) | (pl.col("END_DAY") < 1)
    ),
    "redemption_nonpositive_day": (
        "coupon_redemption", pl.col("DAY") < 1
    ),
    "causal_blank_display": (
        "causal", pl.col("display").is_null() | (pl.col("display").str.strip_chars() == "")
    ),
    "causal_blank_mailer": (
        "causal", pl.col("mailer").is_null() | (pl.col("mailer").str.strip_chars() == "")
    ),
}
domain_summary = pl.concat(
    [
        frame.filter(condition).select(
            pl.lit(rule).alias("rule"),
            pl.lit(table_name).alias("table"),
            pl.len().alias("n_rows"),
        ).collect(engine="streaming")
        for rule, (table_name, condition) in domain_rules.items()
        for frame in [tables[table_name]]
    ]
).sort("n_rows", descending=True)
domain_summary

## 7. Referential integrity giữa các bảng

Household không có demographic được báo là `coverage`, không phải lỗi. Các quan hệ product, campaign và coupon không tồn tại ở bảng cha được xem là lỗi khóa ngoại.

In [ ]:
transaction_products = tables["transaction"].select("PRODUCT_ID").unique()
transaction_households = tables["transaction"].select("household_key").unique()
product_keys = tables["product"].select("PRODUCT_ID").unique()
demographic_keys = tables["demographic"].select("household_key").unique()
campaign_keys = tables["campaign_desc"].select("CAMPAIGN").unique()
coupon_keys = tables["coupon"].select("COUPON_UPC", "CAMPAIGN").unique()
assignment_keys = tables["campaign_table"].select(
    "household_key", "CAMPAIGN"
).unique()

referential_checks = {
    "transaction_product_missing_from_catalog": (
        "invalid_fk",
        transaction_products.join(product_keys, on="PRODUCT_ID", how="anti"),
    ),
    "transaction_household_without_demographic": (
        "coverage",
        transaction_households.join(
            demographic_keys, on="household_key", how="anti"
        ),
    ),
    "campaign_assignment_missing_description": (
        "invalid_fk",
        tables["campaign_table"].select("CAMPAIGN").unique().join(
            campaign_keys, on="CAMPAIGN", how="anti"
        ),
    ),
    "coupon_product_missing_from_catalog": (
        "invalid_fk",
        tables["coupon"].select("PRODUCT_ID").unique().join(
            product_keys, on="PRODUCT_ID", how="anti"
        ),
    ),
    "coupon_campaign_missing_description": (
        "invalid_fk",
        tables["coupon"].select("CAMPAIGN").unique().join(
            campaign_keys, on="CAMPAIGN", how="anti"
        ),
    ),
    "redemption_missing_coupon_campaign_mapping": (
        "invalid_fk",
        tables["coupon_redemption"].select("COUPON_UPC", "CAMPAIGN").unique().join(
            coupon_keys, on=["COUPON_UPC", "CAMPAIGN"], how="anti"
        ),
    ),
    "redemption_household_not_assigned_campaign": (
        "review",
        tables["coupon_redemption"].select("household_key", "CAMPAIGN").unique().join(
            assignment_keys, on=["household_key", "CAMPAIGN"], how="anti"
        ),
    ),
}
referential_summary = pl.concat(
    [
        check.select(
            pl.lit(rule).alias("rule"),
            pl.lit(severity).alias("severity"),
            pl.len().alias("n_keys"),
        ).collect(engine="streaming")
        for rule, (severity, check) in referential_checks.items()
    ]
).sort("n_keys", descending=True)
referential_summary

In [ ]:
redemption_timing_issues = (
    tables["coupon_redemption"]
    .join(
        tables["campaign_desc"].select("CAMPAIGN", "START_DAY", "END_DAY"),
        on="CAMPAIGN",
        how="left",
    )
    .filter(
        pl.col("START_DAY").is_not_null()
        & ~pl.col("DAY").is_between(pl.col("START_DAY"), pl.col("END_DAY"))
    )
)
redemption_timing_issues.select(
    pl.len().alias("n_redemptions_outside_campaign_window")
).collect(engine="streaming")

## 8. Export các bảng audit nhỏ

In [ ]:
quality_dir = project_root / "data" / "processed" / "data_quality"
quality_dir.mkdir(parents=True, exist_ok=True)
quality_overview.write_csv(quality_dir / "table_quality_overview.csv")
null_summary.write_csv(quality_dir / "null_summary.csv")
duplicate_summary.write_csv(quality_dir / "duplicate_summary.csv")
transaction_rule_summary.write_csv(
    quality_dir / "transaction_rule_summary.csv"
)
domain_summary.write_csv(quality_dir / "domain_summary.csv")
referential_summary.write_csv(quality_dir / "referential_summary.csv")
quality_dir

## 9. Cách ra quyết định làm sạch

- Không drop chỉ vì một rule có count lớn; xem sample rows và đối chiếu ý nghĩa nghiệp vụ trước.
- `QUANTITY = 0`, quantity âm và sales âm nên giữ riêng để nhận diện return/adjustment; target purchase chỉ dùng `QUANTITY > 0` và `SALES_VALUE >= 0`.
- `SALES_VALUE = 0` với quantity dương vẫn được giữ là purchase hợp lệ theo quyết định hiện tại.
- Duplicate theo grain cần kiểm tra các cột còn lại: dòng giống hệt có thể deduplicate, nhưng cùng khóa với giá trị khác nhau cần rule hợp nhất cụ thể.
- Thiếu demographic là coverage limitation, không phải lỗi transaction.
- Chỉ materialize clean tables sau khi các rule `review` đã được xác nhận.